# Tutorial 05 — Impact Assessment with bw2calc

Companion explainer: **05_lcia_with_bw2calc.md**. The `LCA` object lifecycle,
choosing methods, and the efficient pattern for many calculations
(factorize once, `switch_method`, `lcia(demand=...)`).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # for bw_helpers
import numpy as np
import pandas as pd
import bw2data as bd
import bw2calc as bc

bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

## Rebuild the kettle system (self-contained) with 2 alternatives
baseline (coal grid) vs a lower-carbon grid variant.

In [2]:
def find_flow(name, categories=("air",)):
    return next(f for f in bio if f["name"] == name and f["categories"] == categories)

co2 = find_flow("Carbon dioxide, fossil")
so2 = find_flow("Sulfur dioxide")
nox = find_flow("Nitrogen oxides")

DB = "t05_kettle"
if DB in bd.databases:
    del bd.databases[DB]

def make(grid_co2):
    return {
        (DB, "elec"): {"name": "electricity", "unit": "kilowatt hour", "exchanges": [
            {"input": (DB, "elec"), "amount": 1.0, "type": "production"},
            {"input": co2.key, "amount": grid_co2, "type": "biosphere"},
            {"input": so2.key, "amount": 0.002, "type": "biosphere"},
            {"input": nox.key, "amount": 0.0018, "type": "biosphere"}]},
        (DB, "steel"): {"name": "steel", "unit": "kilogram", "exchanges": [
            {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
            {"input": (DB, "elec"), "amount": 2.9, "type": "technosphere"},
            {"input": co2.key, "amount": 1.9, "type": "biosphere"}]},
        (DB, "kettle_coal"): {"name": "kettle (coal grid)", "unit": "unit", "exchanges": [
            {"input": (DB, "kettle_coal"), "amount": 1.0, "type": "production"},
            {"input": (DB, "steel"), "amount": 1.2, "type": "technosphere"},
            {"input": (DB, "elec"), "amount": 0.8, "type": "technosphere"}]},
    }

bd.Database(DB).write(make(grid_co2=0.95))
kettle = bd.get_node(database=DB, code="kettle_coal")

13:16:41-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

13:16:41-0400

 [

info     

] 

Vacuuming database            

## Basic lifecycle

In [3]:
gwp = next(m for m in bd.methods
           if "IPCC 2013" in str(m) and "GWP100" in str(m).replace(" ", "")
           and "no LT" not in str(m) and "SLCF" not in str(m))
lca = bc.LCA({kettle: 1}, method=gwp)
lca.lci()
lca.lcia()
print("score:", round(lca.score, 4), bd.Method(gwp).metadata.get("unit"))

score:

6.346

kg CO2-Eq

## Build a method portfolio (climate + acidification + a couple more)

In [4]:
def find_methods(*subs, exclude=("no LT",)):
    out = []
    for m in bd.methods:
        s = str(m).lower()
        if all(x.lower() in s for x in subs) and not any(e.lower() in s for e in exclude):
            out.append(m)
    return out

portfolio = [gwp]
for kw in [("acidification",), ("eutrophication", "freshwater"), ("ozone", "formation")]:
    hits = find_methods("recipe", "midpoint", *kw)
    if hits:
        portfolio.append(hits[0])
print("portfolio:")
for m in portfolio:
    print("   ", m, "->", bd.Method(m).metadata.get("unit"))

portfolio:

('IPCC 2013', 'climate change', 'global warming potential (GWP100)')

->

kg CO2-Eq

('ReCiPe 2016 v1.03, midpoint (E)', 'acidification: terrestrial', 'terrestrial acidification potential (TAP)')

->

kg SO2-Eq

('ReCiPe 2016 v1.03, midpoint (E)', 'eutrophication: freshwater', 'freshwater eutrophication potential (FEP)')

->

kg P-Eq

## Efficient many-calculation pattern: factorize + switch_method + lcia(demand=)

In [5]:
lca = bc.LCA({kettle: 1}, method=portfolio[0])
lca.lci(factorize=True)   # LU factorization cached
lca.lcia()

rows = []
for m in portfolio:
    lca.switch_method(m)
    lca.lcia(demand={kettle.id: 1})
    rows.append({"method": " | ".join(m[-2:]),
                 "unit": bd.Method(m).metadata.get("unit"),
                 "score": lca.score})
df = pd.DataFrame(rows)
df

D:\01code\Projects\SDAI- Ecosystem\Brightway2\.venv\Lib\site-packages\bw2calc\lca.py:250: UserWarning: All values in characterization matrix are zero
  warnings.warn("All values in characterization matrix are zero")


,method,unit,score
0,climate change | global warming potential (GWP...,kg CO2-Eq,6.346000
1,acidification: terrestrial | terrestrial acidi...,kg SO2-Eq,0.011333
2,eutrophication: freshwater | freshwater eutrop...,kg P-Eq,0.000000


## Inspecting internals

In [6]:
print("supply array (nonzero):",
      {bd.get_activity(k)["name"]: round(lca.supply_array[v], 4)
       for k, v in lca.dicts.activity.items()})
print("characterized inventory total:", lca.characterized_inventory.sum())

# by-process contribution = column sums of characterized_inventory
ci = lca.characterized_inventory
by_proc = np.asarray(ci.sum(axis=0)).ravel()
for k, v in lca.dicts.activity.items():
    print(f"   {bd.get_activity(k)['name']:24s} {by_proc[v]: .4f}")

supply array (nonzero):

{'electricity': np.float64(4.28), 'steel': np.float64(1.2), 'kettle (coal grid)': np.float64(1.0)}

characterized inventory total:

0.0

   electricity               0.0000

   steel                     0.0000

   kettle (coal grid)        0.0000

Next: **06 — contribution analysis** turns those column/row sums into a
proper interpretation workflow.